# NB2 — Train MIL head on cached embeddings

**Settings:** GPU T4 x2 / P100, Internet OFF.
**Attach:** dataset `rsna-knee-embeddings-v1` (NB1 output), dataset `rsna-knee-weak-labels` (train_weak_v2.csv), competition data (for train.csv only).
Output: `best.pt` (~25 MB) → publish as dataset `rsna-knee-ckpt-v1`.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score

COMP = Path('/kaggle/input/rsna-knee-abnormality-detection')
EMB = Path('/kaggle/input/rsna-knee-embeddings-v1/embeddings')
LABELS = Path('/kaggle/input/rsna-knee-weak-labels/train_weak_v2.csv')

IGNORE = {'StudyInstanceUID', 'Report'}
WPS = 42          # windows per study in embeddings
HIDDEN = 128
EPOCHS = 3
LR = 1e-3
BATCH = 16
FOLD_FRAC = 0.2   # hold-out fraction for monitoring (gold studies prioritized)
SEED = 42

In [ ]:
class EmbDataset(Dataset):
    def __init__(self, rows, label_cols, emb_dir):
        self.rows = rows.reset_index(drop=True)
        self.label_cols = label_cols
        self.emb_dir = Path(emb_dir)
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        r = self.rows.iloc[i]
        uid = str(r['StudyInstanceUID'])
        f = self.emb_dir / f'{uid}.npz'
        emb = np.load(f)['emb'].astype(np.float32)
        w = emb.shape[0]
        if w >= WPS:
            emb = emb[:WPS]
        else:
            emb = np.concatenate([emb] * int(np.ceil(WPS / w)), axis=0)[:WPS]
        y = r[self.label_cols].to_numpy(dtype=np.float32)
        mask = ~np.isnan(y)
        y = np.nan_to_num(y).astype(np.float32)
        return torch.from_numpy(emb), torch.from_numpy(y), torch.from_numpy(mask), uid

In [ ]:
class GatedAttentionMIL(nn.Module):
    def __init__(self, in_features=384, hidden_dim=128, num_classes=12):
        super().__init__()
        self.attention_v = nn.Sequential(nn.Linear(in_features, hidden_dim), nn.Tanh())
        self.attention_u = nn.Sequential(nn.Linear(in_features, hidden_dim), nn.Sigmoid())
        self.attention_weights = nn.Linear(hidden_dim, 1)
        self.classifier = nn.Linear(in_features, num_classes)
    def forward(self, h):
        v = self.attention_v(h); u = self.attention_u(h)
        a = self.attention_weights(v * u)
        w = torch.softmax(a, dim=1)
        m = torch.sum(w * h, dim=1)
        return self.classifier(m)

In [ ]:
df = pd.read_csv(LABELS)
df_gold = pd.read_csv(COMP / 'train.csv')
label_cols = [c for c in df_gold.columns if c not in IGNORE]
# gold rows are preserved verbatim inside train_weak_v2.csv (labeler contract) — verified locally
df['is_gold'] = df['StudyInstanceUID'].isin(df_gold['StudyInstanceUID'])

rng = np.random.RandomState(SEED)
gold_idx = df.index[df['is_gold']].to_numpy()
rng.shuffle(gold_idx)
n_hold = max(6, int(len(gold_idx) * FOLD_FRAC))
hold_gold = set(gold_idx[:n_hold])

rest_idx = df.index[~df['is_gold']].to_numpy()
rng.shuffle(rest_idx)
hold_rest = set(rest_idx[:int(len(rest_idx) * FOLD_FRAC)])

val_idx = sorted(hold_gold | hold_rest)
tr_idx = [i for i in df.index if i not in hold_gold and i not in hold_rest]
print(f'train {len(tr_idx)}  val {len(val_idx)}  (gold in val: {len(hold_gold)}/{len(gold_idx)})')

In [ ]:
device = 'cuda'
model = GatedAttentionMIL(384, HIDDEN, len(label_cols)).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
bce = nn.BCEWithLogitsLoss(reduction='none')

def loss_fn(logits, y, mask):
    l = bce(logits, y)
    return (l * mask).sum() / mask.sum().clamp(min=1)

tr_loader = DataLoader(EmbDataset(df.loc[tr_idx], label_cols, EMB), batch_size=BATCH, shuffle=True, num_workers=2)
va_loader = DataLoader(EmbDataset(df.loc[val_idx], label_cols, EMB), batch_size=BATCH, shuffle=False, num_workers=2)

best_auc = 0
for ep in range(1, EPOCHS + 1):
    model.train(); tl = 0
    for emb, y, mask, _ in tr_loader:
        emb, y, mask = emb.to(device), y.to(device), mask.to(device)
        opt.zero_grad()
        loss = loss_fn(model(emb), y, mask)
        loss.backward(); opt.step()
        tl += loss.item() * len(emb)
    model.eval(); ys, ps = [], []
    with torch.no_grad():
        for emb, y, mask, _ in va_loader:
            p = torch.sigmoid(model(emb.to(device))).cpu().numpy()
            ys.append(y.numpy()); ps.append(p)
    Y, P = np.concatenate(ys), np.concatenate(ps)
    aucs = []
    for j in range(Y.shape[1]):
        ok = ~np.isnan(Y[:, j])
        if ok.sum() > 4 and len(np.unique(Y[ok, j])) > 1:
            aucs.append(roc_auc_score(Y[ok, j], P[ok, j]))
    macro = float(np.mean(aucs))
    print(f'epoch {ep}: loss {tl/len(tr_idx):.4f} val-macro-AUC {macro:.4f}')
    if macro > best_auc:
        best_auc = macro
        torch.save({'state_dict': model.state_dict(), 'label_cols': label_cols, 'hidden': HIDDEN, 'val_auc': macro}, '/kaggle/working/best.pt')
print('best', best_auc)